# Who Is In

This notebook loads the CSVs we need and gives you placeholders for filtering predictions by `game_week_id` and `player_id`.

In [ ]:
import pandas as pd
from pathlib import Path

paths = {
    'profiles': Path('../CSV/13.08.2026/profiles_rows.csv'),
    'predictions': Path('../CSV/24.09.2026/predictions_rows.csv'),
    'fixtures': Path('../CSV/24.09.2026/fixtures_rows.csv'),
    'game_weeks': Path('../CSV/24.09.2026/game_weeks_rows.csv'),
    'season_players': Path('../CSV/13.08.2026/season_players_rows.csv'),
}

def load_and_clean(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    df = pd.read_csv(p, dtype=str)
    df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ('date', 'time', 'created_at', 'kick', 'kickoff', 'updated_at', 'predictions_close')):
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

profiles = load_and_clean(paths['profiles'])
predictions = load_and_clean(paths['predictions'])
fixtures = load_and_clean(paths['fixtures'])
game_weeks = load_and_clean(paths['game_weeks'])
season_players = load_and_clean(paths['season_players'])

print('profiles', profiles.shape)
print('predictions', predictions.shape)
print('fixtures', fixtures.shape)
print('game_weeks', game_weeks.shape)
print('season_players', season_players.shape)

profiles (59, 4)
predictions (14340, 7)
fixtures (410, 8)
game_weeks (41, 8)
season_players (78, 4)


In [ ]:
# Target season and game week placeholders.
# If target_season_id is left blank, it will be derived from the target game week.
target_season_id = 'f010783f-7fb5-40d8-9e00-6d8a45fa448e'
target_game_week_id = '292fe5d7-319d-4d47-9c2b-cd64bdacc205'

from IPython.display import display

if not target_game_week_id:
    raise ValueError('Set target_game_week_id to a valid game_week id')

gw_row = game_weeks[game_weeks['id'].astype(str) == str(target_game_week_id)]
if gw_row.empty:
    raise KeyError(f'Game week id {target_game_week_id} not found in game_weeks')

gw = gw_row.iloc[0]
resolved_season_id = target_season_id or str(gw.get('season_id', ''))
if not resolved_season_id or resolved_season_id == 'nan':
    raise KeyError('Could not resolve a season id. Set target_season_id or choose a game week with a season_id.')

season_roster = season_players[season_players['season_id'].astype(str) == str(resolved_season_id)].copy()
season_roster = season_roster.drop_duplicates(subset=['player_id'])
season_roster = season_roster.merge(
    profiles[['id', 'username']],
    left_on='player_id',
    right_on='id',
    how='left',
    suffixes=('', '_profile')
)

season_fixture_ids = fixtures[
    fixtures['game_week_id'].astype(str).isin(
        game_weeks[game_weeks['season_id'].astype(str) == str(resolved_season_id)]['id'].astype(str)
    )
]['id'].astype(str)

season_predictions = predictions[predictions['fixture_id'].astype(str).isin(season_fixture_ids)].copy()
season_submitters = set(season_predictions['user_id'].dropna().astype(str).unique())
season_roster['has_entered_any_score'] = season_roster['player_id'].astype(str).isin(season_submitters)
season_missing_all = season_roster[~season_roster['has_entered_any_score']].copy()

week_fixture_ids = fixtures[fixtures['game_week_id'].astype(str) == str(target_game_week_id)]['id'].astype(str)
week_predictions = predictions[predictions['fixture_id'].astype(str).isin(week_fixture_ids)].copy()
week_submitters = set(week_predictions['user_id'].dropna().astype(str).unique())
season_roster['has_entered_target_week'] = season_roster['player_id'].astype(str).isin(week_submitters)
week_missing = season_roster[~season_roster['has_entered_target_week']].copy()

summary = pd.DataFrame([{
    'season_id': resolved_season_id,
    'game_week_id': target_game_week_id,
    'season_players': season_roster.shape[0],
    'season_players_with_any_score': int(season_roster['has_entered_any_score'].sum()),
    'season_players_missing_any_score': season_missing_all.shape[0],
    'players_missing_target_week': week_missing.shape[0],
}])

print('Summary:')
display(summary)

print('Players missing any score in this season:')
display(
    season_missing_all[[c for c in ['player_id', 'username', 'created_at'] if c in season_missing_all.columns]]
    .sort_values(by=[c for c in ['username', 'player_id'] if c in season_missing_all.columns])
    .reset_index(drop=True)
)

print('Players missing predictions for the target game week:')
display(
    week_missing[[c for c in ['player_id', 'username', 'created_at'] if c in week_missing.columns]]
    .sort_values(by=[c for c in ['username', 'player_id'] if c in week_missing.columns])
    .reset_index(drop=True)
)

print('If you want to target a different season, set target_season_id. If you want a different week, change target_game_week_id.')

Summary:


,season_id,game_week_id,season_players,season_players_with_any_score,season_players_missing_any_score,players_missing_target_week
0,f010783f-7fb5-40d8-9e00-6d8a45fa448e,ab68800f-a193-4c00-b898-55fd07623d4e,44,44,0,1


Players missing any score in this season:


,player_id,username,created_at


Players missing predictions for the target game week:


,player_id,username,created_at
0,298651b6-b5d4-4875-a636-799edae79a84,KAV,2026-08-19 13:24:05.054246


If you want to target a different season, set target_season_id. If you want a different week, change target_game_week_id.
